In [ ]:
# This Python 3 environment comes with many helpful analytics libraries installed
# It is defined by the kaggle/python Docker image: https://github.com/kaggle/docker-python
# For example, here's several helpful packages to load

import numpy as np # linear algebra
import pandas as pd # data processing, CSV file I/O (e.g. pd.read_csv)
from torch.utils.data import DataLoader
from torch.utils.data import Dataset
import torch


# Input data files are available in the read-only "../input/" directory
# For example, running this (by clicking run or pressing Shift+Enter) will list all files under the input directory

import os
for dirname, _, filenames in os.walk('/kaggle/input/digit_recognizer'):
    for filename in filenames:
        print(os.path.join(dirname, filename))

# You can write up to 20GB to the current directory (/kaggle/working/) that gets preserved as output when you create a version using "Save & Run All" 
# You can also write temporary files to /kaggle/temp/, but they won't be saved outside of the current session

In [ ]:
df = pd.read_csv("/kaggle/input/digit-recognizer/train.csv")


In [ ]:
from sklearn .model_selection import train_test_split

train_df, test_df = train_test_split(df, train_size=0.8, random_state=42)
train_df = train_df.to_numpy()
test_df = test_df.to_numpy()
y_train, x_train = train_df[:,0], train_df[:,1:].astype(np.float32)
y_test, x_test = test_df[:,0], test_df[:,1:].astype(np.float32)
device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')



In [ ]:
x_train, y_train, x_test, y_test = map(torch.tensor,(x_train, y_train, x_test, y_test))

In [ ]:
x_train,x_test= x_train/255, x_test/255,


In [ ]:
eps = 1e-6
def relu(x):
    return x.clamp(min=0)

def log_softmax(x):
    return x - torch.log(torch.exp(x).sum(dim=1, keepdim=True))

def nll_loss(log_probs, targets):
    batch_size = log_probs.shape[0]
    return -log_probs[range(batch_size), targets].mean()

def batch_norm(x, gamma):
    batch_mean = x.mean(dim=0)
    batch_var = (x-batch_mean).var(dim=0)
    x_norm = (x-batch_mean)/torch.sqrt(batch_var + eps)
    return gamma + x_norm
    

In [ ]:
BATCH_SIZE = 64
EPOCH = 300
EPOCH_PER_EVAL = 10
lr = 1e-2
hidden_features = 512

In [ ]:
x_train, y_train, x_test, y_test = [t.to(device) for t in [x_train, y_train, x_test, y_test]]
w1 = torch.distributions.Uniform(-np.sqrt(1/784), np.sqrt(1/784)).sample((hidden_features, 784)).to(device).requires_grad_()
b1 = torch.distributions.Uniform(-np.sqrt(1/784), np.sqrt(1/784)).sample((hidden_features, )).to(device).requires_grad_()
w2 = torch.distributions.Uniform(-np.sqrt(1/hidden_features), np.sqrt(1/hidden_features)).sample((hidden_features, hidden_features)).to(device).requires_grad_()
b2 = torch.distributions.Uniform(-np.sqrt(1/hidden_features), np.sqrt(1/hidden_features)).sample((hidden_features, )).to(device).requires_grad_()
w3 = torch.distributions.Uniform(-np.sqrt(1/hidden_features), np.sqrt(1/hidden_features)).sample((10, hidden_features)).to(device).requires_grad_()
b3 = torch.distributions.Uniform(-np.sqrt(1/hidden_features), np.sqrt(1/hidden_features)).sample((10, )).to(device).requires_grad_()

weights = [w1,w2,w3]
bias = [b1,b2,b3]

In [ ]:
class MNIST(Dataset):
    def __init__(self,x,y):
        self.x = x
        self.y = y
        self.n = len(x)

    def __len__(self):
        return self.n
    def __getitem__(self,idx):
        return self.x[idx], self.y[idx]

train_ds = MNIST(x_train, y_train)
test_ds = MNIST(x_test, y_test)
train_dl = DataLoader(train_ds,batch_size = BATCH_SIZE)
test_dl = DataLoader(test_ds,batch_size = BATCH_SIZE)


In [ ]:
def model(input, weights, bias):
    temp = input
    for i in range(len(weights)-1):
        temp = relu(temp @ weights[i].t() + bias[i])
    temp = temp @ weights[-1].t() + bias[-1]
    return log_softmax(temp)

In [ ]:
lp = model(train_ds.x[0].unsqueeze(0),weights,bias)
nll_loss(lp,train_ds.y)

In [ ]:
for epoch in range(EPOCH):
    for i, (xb, yb) in enumerate(train_dl):
        
        xb, yb = xb.to(device), yb.to(device)
        log_probs = model(xb, weights, bias)
        loss = nll_loss(log_probs, yb)
        loss.backward()
        
        with torch.no_grad(): 
            for param in weights:
                param -= param.grad * lr
            for param in bias:
                param -= param.grad * lr
                
            for param in weights:
                param.grad.zero_()
            for param in bias:
                param.grad.zero_()
    

    if (epoch + 1) % EPOCH_PER_EVAL == 0:
        total_correct = 0
        total_samples = 0
        
        with torch.no_grad():
            for xb_test, yb_test in test_dl:
                xb_test, yb_test = xb_test.to(device), yb_test.to(device)
                
                log_probs = model(xb_test, weights, bias)
                preds = torch.argmax(log_probs, dim=1)
                total_samples += yb_test.size(0)
                total_correct += (preds == yb_test).sum().item()
        
        accuracy = 100 * total_correct / total_samples
        print(f'Epoch [{epoch+1}/{EPOCH}], Loss: {loss.item():.4f}, Test Accuracy: {accuracy:.2f}%')